In [2]:
!pip install fuzzywuzzy

In [3]:
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import pandas as pd
import numpy as np
from collections import OrderedDict
import scipy.stats as stats

/usr/local/lib/python3.12/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [4]:
def extractLastNames(names):
    lastNames = []
    for name in names:
      parts = name.strip().lower().split()
      lastNames.append(parts[-1].strip())
    return lastNames

In [5]:
thresholds = [60, 70, 80, 90]

In [6]:
df = pd.read_csv("researchers.csv")
df.head()

,Name,Google Scholar ID,OpenAlex URL,Citation Count,Publication Count,Email Domain,Affiliation,Field,Subfield,Citation Level,...,Region,Co-author count (Google Scholar),Co-authors' names (Google Scholar),Co-authors' names (OpenAlex),Co-authors' names (DeepSeek R1),Reality (DeepSeek R1),Co-authors' names (Llama 4 Scout),Reality (Llama 4 Scout),Co-authors' names (Mixtral),Reality (Mixtral)
0,Kuttanelloor Roshni,xvwC5SMAAAAJ,https://openalex.org/a5049943816,163,29,@cusat.ac.in,"Post Doctoral Fellow, School of Environmental ...","Agriculture, Fisheries & Forestry",Fisheries,Low,...,East/South-East Asia,3,K Ranjeet/Neelesh Dahanukar/Rajeev Raghavan,B Madhusoodana Kurup/Balakrishna Madhusoodana ...,Shashi Bhushan/Vinay Kumar Vase/Achamveetil Go...,-,NaN,-,S. Sree Ranjani/S. Jegadhesan/R. Anbazhagan/K ...,-
1,Siriluck Thammanu,H6zh8jwAAAAJ,https://openalex.org/A5090976916,120,8,@mail.forest.go.th,"Royal Forest Department, Thailand","Agriculture, Fisheries & Forestry",Forestry,Low,...,East/South-East Asia,3,Jamroon Srichaichana/Dokrak Marod/Hee Han,Akbar I. Inamdar/Arjun Magotra/Atchara Teerawa...,Harald Vacik/Eakaphong Ashraf/Shinya Numata,-,NaN,-,Pattarapong Kongcharoen/Kitiyos Srisuksanti/Su...,-
2,M.S.M. Nafees,eQr4PLYAAAAJ,https://openalex.org/a5079244643,113,15,@esn.ac.lk,Senior Lecturer,"Agriculture, Fisheries & Forestry",Dairy & Animal Science,Low,...,East/South-East Asia,3,A.R.S.B. Athauda/Saman Athauda/M Pagthinathan/...,A. R. S. B. Athauda/Clement R. de Cruz/E. Pows...,Qamar Bilal/Aisha Khatoon/Muhammad Tahir,-,NaN,-,Muhammad Farrukh Shahid/Abdul Rauf/Muhammad Im...,-
3,Aman Kumar Gupta,cab6DuAAAAAJ,https://openalex.org/a5101497736,122,17,NaN,Banaras Hindu University,"Agriculture, Fisheries & Forestry",Agronomy & Agriculture,Low,...,East/South-East Asia,3,Poulomi Dey/Jiwan Paudel/Ashish Chaudhary,Akshay Chaudhary/Alberto Sanz-Cobeña/Alice Min...,Rajendra Prasad/Anil Kumar Singh/Shiv Datt Sharma,-,NaN,-,Surjeet Singh/Jaspreet Singh/Amrik S. Basra,-
4,Seralathan Kamala‐Kannan,ERXuIzUAAAAJ,https://openalex.org/A5075977093,158,178,@annamalaiuniversity.ac.in,"Department of Horticulture, Faculty of Agricul...","Agriculture, Fisheries & Forestry",Horticulture,Low,...,East/South-East Asia,5,S. Venkatesan/T.Uma Maheswari/S Kumar/R. Sudha...,A. Bedoui/A. Deepak/A. K. Ramasamy/A. Kanakala...,K. Prabakar/R. Samiyappan/A. Nithya/M. Senthil...,-,NaN,-,Ramalingam Palanisamy/Arulmozhi Marimuthu/Siva...,-


In [7]:
df = df[df["Reality (Mixtral)"] != "hypothetical"]
df = df[df["Co-authors' names (Mixtral)"].notna()]
df = df[df["Co-authors' names (OpenAlex)"].notna()]

In [8]:
len(df)

1510

In [9]:
df["Match Count 60%"] = 0
df["Match Count 70%"] = 0
df["Match Count 80%"] = 0
df["Match Count 90%"] = 0

In [10]:
for index, row in df.iterrows():
    llmNames = row["Co-authors' names (Mixtral)"].split('/')
    refNames = row["Co-authors' names (OpenAlex)"].split('/')

    llmNames = extractLastNames(llmNames)
    refNames = extractLastNames(refNames)

    for threshold in thresholds:
        count = 0
        refNamesCopy = refNames.copy()

        for name in llmNames:
            if not refNamesCopy:
                break

            bestRatio = 0
            bestIndex = -1
            for i, candidate in enumerate(refNamesCopy):
                currentRatio = fuzz.ratio(name, candidate)
                if currentRatio >= threshold and currentRatio > bestRatio:
                    bestRatio = currentRatio
                    bestIndex = i

            if bestIndex != -1:
                count += 1
                del refNamesCopy[bestIndex]

        colName = "Match Count " + str(threshold) + "%"
        df.at[index, colName] = count

In [11]:
df.head()

,Name,Google Scholar ID,OpenAlex URL,Citation Count,Publication Count,Email Domain,Affiliation,Field,Subfield,Citation Level,...,Co-authors' names (DeepSeek R1),Reality (DeepSeek R1),Co-authors' names (Llama 4 Scout),Reality (Llama 4 Scout),Co-authors' names (Mixtral),Reality (Mixtral),Match Count 60%,Match Count 70%,Match Count 80%,Match Count 90%
0,Kuttanelloor Roshni,xvwC5SMAAAAJ,https://openalex.org/a5049943816,163,29,@cusat.ac.in,"Post Doctoral Fellow, School of Environmental ...","Agriculture, Fisheries & Forestry",Fisheries,Low,...,Shashi Bhushan/Vinay Kumar Vase/Achamveetil Go...,-,NaN,-,S. Sree Ranjani/S. Jegadhesan/R. Anbazhagan/K ...,-,1,1,1,1
1,Siriluck Thammanu,H6zh8jwAAAAJ,https://openalex.org/A5090976916,120,8,@mail.forest.go.th,"Royal Forest Department, Thailand","Agriculture, Fisheries & Forestry",Forestry,Low,...,Harald Vacik/Eakaphong Ashraf/Shinya Numata,-,NaN,-,Pattarapong Kongcharoen/Kitiyos Srisuksanti/Su...,-,0,0,0,0
2,M.S.M. Nafees,eQr4PLYAAAAJ,https://openalex.org/a5079244643,113,15,@esn.ac.lk,Senior Lecturer,"Agriculture, Fisheries & Forestry",Dairy & Animal Science,Low,...,Qamar Bilal/Aisha Khatoon/Muhammad Tahir,-,NaN,-,Muhammad Farrukh Shahid/Abdul Rauf/Muhammad Im...,-,0,0,0,0
3,Aman Kumar Gupta,cab6DuAAAAAJ,https://openalex.org/a5101497736,122,17,NaN,Banaras Hindu University,"Agriculture, Fisheries & Forestry",Agronomy & Agriculture,Low,...,Rajendra Prasad/Anil Kumar Singh/Shiv Datt Sharma,-,NaN,-,Surjeet Singh/Jaspreet Singh/Amrik S. Basra,-,2,1,0,0
4,Seralathan Kamala‐Kannan,ERXuIzUAAAAJ,https://openalex.org/A5075977093,158,178,@annamalaiuniversity.ac.in,"Department of Horticulture, Faculty of Agricul...","Agriculture, Fisheries & Forestry",Horticulture,Low,...,K. Prabakar/R. Samiyappan/A. Nithya/M. Senthil...,-,NaN,-,Ramalingam Palanisamy/Arulmozhi Marimuthu/Siva...,-,4,3,2,2


In [12]:
df["DNE 60%"] = 0
df["DNE 70%"] = 0
df["DNE 80%"] = 0
df["DNE 90%"] = 0

for index, row in df.iterrows():

    N = len(row["Co-authors' names (Google Scholar)"].split("/"))
    R = len(row["Co-authors' names (OpenAlex)"].split("/"))

    for threshold in [60, 70, 80, 90]:
        match_col = f"Match Count {threshold}%"
        DNE_col = f"DNE {threshold}%"

        denom = max(1, min(N, R))
        DNE = min(row[match_col] / denom, 1.0)

        df.at[index, DNE_col] = DNE

/tmp/ipython-input-2300685900.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.3333333333333333' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[index, DNE_col] = DNE
/tmp/ipython-input-2300685900.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.3333333333333333' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[index, DNE_col] = DNE
/tmp/ipython-input-2300685900.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.3333333333333333' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[index, DNE_col] = DNE
/tmp/ipython-input-2300685900.py:18: FutureWarning: Setting an item of incompatible dtype i

In [13]:
print("Mean DNE", np.mean(df["DNE 60%"]))
print("Std DNE", df["DNE 60%"].std())

Mean DNE 0.49232559831406614
Std DNE 0.35752135197789064


In [14]:
fields = list(df["Field"].unique())
fields

['Agriculture, Fisheries & Forestry',
 'Biology',
 'Built Environment & Design',
 'Clinical Medicine',
 'Earth & Environmental Sciences',
 'Economics & Business',
 'Engineering',
 'Information & Communication Technologies',
 'Mathematics & Statistics',
 'Physics & Astronomy']

In [15]:
metrics = ["DNE 60%", "DNE 70%", "DNE 80%", "DNE 90%"]

levels = ['High', 'Low']

In [16]:
regions = list(df["Region"].unique())
regions

['East/South-East Asia',
 'Europe',
 'Middle East',
 'North Africa',
 'North America',
 'Oceanic',
 'South/Central America',
 'Sub-Saharan Africa']

# Mean DNE without disaggregation

In [17]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for metric in metrics:
      print(metric, "*****", sf[metric].mean())

Agriculture, Fisheries & Forestry
DNE 60% ***** 0.5458374972039454
DNE 70% ***** 0.303555734731837
DNE 80% ***** 0.2216927117528062
DNE 90% ***** 0.17509561771072482
Biology
DNE 60% ***** 0.5047411816130802
DNE 70% ***** 0.24633690920574272
DNE 80% ***** 0.1488109127730837
DNE 90% ***** 0.10620702052953629
Built Environment & Design
DNE 60% ***** 0.3816339478501627
DNE 70% ***** 0.203705209130862
DNE 80% ***** 0.12442612943657771
DNE 90% ***** 0.09701383265759148
Clinical Medicine
DNE 60% ***** 0.6346396532188191
DNE 70% ***** 0.37004169730808295
DNE 80% ***** 0.24522643583087292
DNE 90% ***** 0.1764971023094277
Earth & Environmental Sciences
DNE 60% ***** 0.5310399636194595
DNE 70% ***** 0.31589659228497996
DNE 80% ***** 0.22718022940719534
DNE 90% ***** 0.18217431156885114
Economics & Business
DNE 60% ***** 0.37227083723157545
DNE 70% ***** 0.175089317753657
DNE 80% ***** 0.11728686972487208
DNE 90% ***** 0.07400026984212138
Engineering
DNE 60% ***** 0.46983340766164194
DNE 70% *****

In [18]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for metric in metrics:
    print(metric, "*****", sf[metric].mean())

East/South-East Asia
DNE 60% ***** 0.6084862379090523
DNE 70% ***** 0.43376540256310814
DNE 80% ***** 0.3586214131041655
DNE 90% ***** 0.31404905547570694
Europe
DNE 60% ***** 0.47133980152478727
DNE 70% ***** 0.22790179006982897
DNE 80% ***** 0.14007642889313857
DNE 90% ***** 0.09856421930197587
Middle East
DNE 60% ***** 0.45733836461659105
DNE 70% ***** 0.22687469869956353
DNE 80% ***** 0.13264484881268
DNE 90% ***** 0.0872377112366682
North Africa
DNE 60% ***** 0.5188531941208299
DNE 70% ***** 0.2491019995459941
DNE 80% ***** 0.15372589395078645
DNE 90% ***** 0.0979515432721672
North America
DNE 60% ***** 0.5096707960804903
DNE 70% ***** 0.29603627159143264
DNE 80% ***** 0.2314995119019912
DNE 90% ***** 0.17629391641051997
Oceanic
DNE 60% ***** 0.5412152395607532
DNE 70% ***** 0.30998445907708766
DNE 80% ***** 0.21876891618990418
DNE 90% ***** 0.16740311932738344
South/Central America
DNE 60% ***** 0.4257696541700231
DNE 70% ***** 0.2445883348545388
DNE 80% ***** 0.17894004555379248

# Mean DNE disaggregated by Citation Level

In [19]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for level in levels:
    print(level)
    hf = sf[sf["Citation Level"] == level]
    for metric in metrics:
      print(metric, "*****", hf[metric].mean())

Agriculture, Fisheries & Forestry
High
DNE 60% ***** 0.6969270131235556
DNE 70% ***** 0.4060627272749975
DNE 80% ***** 0.2842781857995438
DNE 90% ***** 0.2300709830952319
Low
DNE 60% ***** 0.39474798128433497
DNE 70% ***** 0.2010487421886764
DNE 80% ***** 0.15910723770606855
DNE 90% ***** 0.12012025232621772
Biology
High
DNE 60% ***** 0.6730919189694923
DNE 70% ***** 0.3872177373691243
DNE 80% ***** 0.24524199150916823
DNE 90% ***** 0.17222188489953524
Low
DNE 60% ***** 0.33171403488565676
DNE 70% ***** 0.10154272470448941
DNE 80% ***** 0.04970119296099688
DNE 90% ***** 0.03835840992703738
Built Environment & Design
High
DNE 60% ***** 0.49414805458670397
DNE 70% ***** 0.2595394342786404
DNE 80% ***** 0.17738312128490943
DNE 90% ***** 0.13642337181327377
Low
DNE 60% ***** 0.261196312470203
DNE 70% ***** 0.14393899629662027
DNE 80% ***** 0.06773977196512407
DNE 90% ***** 0.05482897384305835
Clinical Medicine
High
DNE 60% ***** 0.7606893840900147
DNE 70% ***** 0.48282598305829183
DNE 80% 

In [20]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for level in levels:
    print(level)
    hf = sf[sf["Citation Level"] == level]
    for metric in metrics:
      print(metric, "*****", hf[metric].mean())

East/South-East Asia
High
DNE 60% ***** 0.7803156769671654
DNE 70% ***** 0.6154772993739384
DNE 80% ***** 0.5218221227883462
DNE 90% ***** 0.45580458068177315
Low
DNE 60% ***** 0.4291859536744996
DNE 70% ***** 0.24415298849963313
DNE 80% ***** 0.18832502039023777
DNE 90% ***** 0.1661302465650292
Europe
High
DNE 60% ***** 0.5904011652388049
DNE 70% ***** 0.3028224536655717
DNE 80% ***** 0.1888061583315361
DNE 90% ***** 0.13891209903453014
Low
DNE 60% ***** 0.3457366046396697
DNE 70% ***** 0.14886460649629815
DNE 80% ***** 0.08866924179329067
DNE 90% ***** 0.05599942310059996
Middle East
High
DNE 60% ***** 0.6052634993983078
DNE 70% ***** 0.3152463475638171
DNE 80% ***** 0.1956115857505456
DNE 90% ***** 0.12739980005799934
Low
DNE 60% ***** 0.30633145619358854
DNE 70% ***** 0.1366619738173047
DNE 80% ***** 0.06836630485527544
DNE 90% ***** 0.04623891223155929
North Africa
High
DNE 60% ***** 0.5943802820517546
DNE 70% ***** 0.32109130755667076
DNE 80% ***** 0.20552753189813636
DNE 90% ***

# Mean Comparison

In [21]:
lowCitedDNE = df[df["Citation Level"] == "Low"]["DNE 60%"].to_list()
highlyCitedDNE = df[df["Citation Level"] == "High"]["DNE 60%"].to_list()

In [22]:
print("Low cited Avg DNE:", np.mean(lowCitedDNE))
print("highly cited Avg DNE:", np.mean(highlyCitedDNE))

Low cited Avg DNE: 0.34990860794196077
highly cited Avg DNE: 0.6317561249300067


In [23]:
t_stat, p_value = stats.ttest_ind(highlyCitedDNE, lowCitedDNE, alternative='greater')

In [24]:
print(f"T-statistic: {t_stat}")
print(f"P-value: {p_value}")

T-statistic: 16.660608552326813
P-value: 1.2247647074951991e-57


## T-Test

In [25]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    t_stat, p_value = stats.ttest_ind(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

Agriculture, Fisheries & Forestry
DNE 60% ***** 3.2595171468946695e-08
DNE 70% ***** 3.124743806234925e-05
DNE 80% ***** 0.005160618070969239
DNE 90% ***** 0.008207422971142431
Biology
DNE 60% ***** 1.4280127698986163e-10
DNE 70% ***** 2.1422402174740677e-10
DNE 80% ***** 1.0498047276733144e-07
DNE 90% ***** 1.6996574441347464e-05
Built Environment & Design
DNE 60% ***** 4.703999514291641e-06
DNE 70% ***** 0.004846211150077819
DNE 80% ***** 0.0008456583231230718
DNE 90% ***** 0.007245421146479827
Clinical Medicine
DNE 60% ***** 4.2846844428087485e-07
DNE 70% ***** 3.3617606910743597e-06
DNE 80% ***** 8.635670633654333e-05
DNE 90% ***** 0.0008167357160877519
Earth & Environmental Sciences
DNE 60% ***** 1.2885954270353137e-11
DNE 70% ***** 2.453069403247489e-07
DNE 80% ***** 4.7205057007956914e-05
DNE 90% ***** 0.0010470573432763406
Economics & Business
DNE 60% ***** 0.0045173081032398345
DNE 70% ***** 0.11889652069823713
DNE 80% ***** 0.4669781349158017
DNE 90% ***** 0.25542766119170845

In [26]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    t_stat, p_value = stats.ttest_ind(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

East/South-East Asia
DNE 60% ***** 3.212446478856272e-12
DNE 70% ***** 6.245412352228108e-12
DNE 80% ***** 3.6481542330100827e-10
DNE 90% ***** 2.672166653858614e-08
Europe
DNE 60% ***** 9.489184442486303e-08
DNE 70% ***** 2.4286326996647148e-05
DNE 80% ***** 0.0010119816620376843
DNE 90% ***** 0.002795922141868542
Middle East
DNE 60% ***** 2.3906256497038295e-10
DNE 70% ***** 6.315220969583793e-07
DNE 80% ***** 1.1746314053302755e-05
DNE 90% ***** 0.0013597169479153402
North Africa
DNE 60% ***** 0.002635799628834004
DNE 70% ***** 0.0004120133050122633
DNE 80% ***** 0.0019417482392999972
DNE 90% ***** 0.00526438939661624
North America
DNE 60% ***** 3.5243826177475854e-08
DNE 70% ***** 3.075128358320098e-05
DNE 80% ***** 0.0009019125294229748
DNE 90% ***** 0.003087922955658022
Oceanic
DNE 60% ***** 2.230583823233941e-12
DNE 70% ***** 2.186288740435793e-05
DNE 80% ***** 0.004936115039748231
DNE 90% ***** 0.017941898304474797
South/Central America
DNE 60% ***** 5.27411092870662e-13
DNE 70

## Mann-Whitney U Test

In [27]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    u_stat, p_value = stats.mannwhitneyu(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

Agriculture, Fisheries & Forestry
DNE 60% ***** 5.689003586159955e-07
DNE 70% ***** 3.5365453982977296e-06
DNE 80% ***** 0.00022292205410079412
DNE 90% ***** 0.00025081892021089884
Biology
DNE 60% ***** 3.3841329933064658e-09
DNE 70% ***** 1.4855355886670044e-12
DNE 80% ***** 3.6376628343436513e-12
DNE 90% ***** 2.5667906078076124e-08
Built Environment & Design
DNE 60% ***** 8.619254151899583e-07
DNE 70% ***** 2.1836864963961667e-05
DNE 80% ***** 3.260758589610466e-08
DNE 90% ***** 5.6003801005034184e-08
Clinical Medicine
DNE 60% ***** 1.244185774493359e-06
DNE 70% ***** 2.109657452609894e-06
DNE 80% ***** 1.1204463709994605e-05
DNE 90% ***** 3.5319820412091844e-05
Earth & Environmental Sciences
DNE 60% ***** 5.330612392528362e-10
DNE 70% ***** 3.350290681734218e-09
DNE 80% ***** 2.2765674361237386e-08
DNE 90% ***** 8.904118840835724e-07
Economics & Business
DNE 60% ***** 0.0003349733270367767
DNE 70% ***** 9.493493838237529e-05
DNE 80% ***** 0.004942720824743358
DNE 90% ***** 0.000486

In [28]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    u_stat, p_value = stats.mannwhitneyu(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

East/South-East Asia
DNE 60% ***** 1.2795793227453247e-10
DNE 70% ***** 1.7643882899935043e-10
DNE 80% ***** 1.9485195379498517e-09
DNE 90% ***** 1.5320267676168296e-08
Europe
DNE 60% ***** 2.5043189269164913e-07
DNE 70% ***** 1.2935170312230744e-06
DNE 80% ***** 1.7970107122195266e-06
DNE 90% ***** 1.0232178329685497e-06
Middle East
DNE 60% ***** 2.220751015155253e-10
DNE 70% ***** 4.969574774454151e-09
DNE 80% ***** 1.9497332184311038e-10
DNE 90% ***** 5.664666971215193e-10
North Africa
DNE 60% ***** 0.006304463904877779
DNE 70% ***** 1.1741938517386595e-06
DNE 80% ***** 1.8069009333722684e-08
DNE 90% ***** 1.7301127425754844e-07
North America
DNE 60% ***** 1.427122818265989e-07
DNE 70% ***** 3.894418935158614e-09
DNE 80% ***** 2.767648414868108e-08
DNE 90% ***** 9.689049432110817e-08
Oceanic
DNE 60% ***** 5.75267025511638e-10
DNE 70% ***** 3.696064938442941e-07
DNE 80% ***** 4.969785934129907e-06
DNE 90% ***** 1.623887111948236e-05
South/Central America
DNE 60% ***** 3.6526526680312